# 🎯 Module 13: Fine-Tuning and Open Models

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 3 — LLM Application Engineering  
> **Module:** 13 — Fine-Tuning and Open Models  
> **Planned Date:** August 15, 2026  
> **Level:** Intermediate → Advanced  
> **Prerequisites:** LLM Fundamentals, Python, Deep Learning, Evaluation & Guardrails  
> **Previous:** 12 — LLM Evaluation and Guardrails  
> **Next:** 14 — Multimodal GenAI

---

Now we **customize** open-weight language models.

In this module we'll cover:

- What **fine-tuning** is (and isn't)
- **Hosted APIs vs open-weight** models
- **When** to fine-tune (and when NOT to)
- Preparing a **training dataset**
- **Tokenization** and **chat templates**
- **Full fine-tuning vs LoRA vs QLoRA**
- The **Hugging Face ecosystem** (Transformers, Datasets, PEFT, TRL)
- A **practical LoRA lab**
- **Evaluating** against a baseline
- **Data, licensing, and safety**
- A **final project**: Domain-Specific Fine-Tuned Assistant

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain **pretrained, instruction-tuned, and fine-tuned** models
2. Distinguish **open-weight** models from fully open-source
3. Compare **hosted** vs **local** model inference
4. Decide whether **prompting, RAG, or fine-tuning** best fits a problem
5. Prepare and **validate** an instruction-tuning dataset
6. Understand **tokenization, labels, causal LM training**
7. Explain **full fine-tuning, LoRA, QLoRA**
8. Understand the **Hugging Face ecosystem**
9. Run a small **supervised fine-tuning** experiment
10. **Evaluate** the fine-tuned model against a **baseline**
11. Understand **adapter storage, licensing, hardware limits, deployment trade-offs**

---
## ⚙️ 0. Setup & Imports

This notebook is **safe to run without a GPU or API key**. All the fine-tuning code is presented as **templates** and can be executed when you have the required hardware and libraries installed.

In [ ]:
import os
import json
from pathlib import Path

print('✅ Setup ready.')
print('Working directory:', Path.cwd())

# Check available libraries (any missing is fine — the code is shown as templates)
for pkg in ['torch', 'transformers', 'datasets', 'peft', 'trl', 'accelerate']:
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f'✅ {pkg:12s} → {version}')
    except ImportError:
        print(f'❌ {pkg:12s} → not installed')

---
## 🧠 1. What Is Fine-Tuning?

**Fine-tuning** = taking a **pretrained** model and continuing its training on a **selected dataset** to adapt its behavior to a task or style.

### Fine-tuning tasks

- Following a particular instruction format
- Consistent support-ticket classification
- Writing responses in a defined style
- Extracting info in a predictable format
- Adapting to specialized terminology
- Improving performance on a specific task

> ⚠️ **Fine-tuning ≠ training from scratch.** Training from scratch requires far more data, compute, and engineering.

### The customization pipeline

```text
Pretrained Model
       │
       ▼
Prepare High-Quality Examples
       │
       ▼
Choose Training Method
       │
       ▼
Fine-Tune Model or Adapter
       │
       ▼
Evaluate on Held-Out Data
       │
       ▼
Compare With Baseline
       │
       ▼
Keep, Improve, or Reject the Result
```

> ⚠️ **Fine-tuning does NOT guarantee better performance.** Poor data or bad settings can make a model **worse**.

---
## 🧩 2. Hosted APIs vs Open-Weight Models

> ⚠️ **"Open weights" ≠ "open source" ≠ "unrestricted use".** Check the **license, acceptable-use terms, and redistribution requirements**.

| Feature | Hosted API | Open-Weight |
| --- | --- | --- |
| Hosting | Provider operates | You or a service provider |
| Infrastructure | Handled | Depends on hosting |
| Customization | Provider-dependent | Custom inference & tuning |
| Setup | Straightforward | May need more infra |
| Hardware | Abstracted | Your responsibility |
| Data handling | Provider terms | Your deployment |
| Cost | Usage-based | Infra + storage + ops |
| Access | Provider-controlled | License-controlled |

### Model-Hub concepts

- **Model ID / repo ID**
- **Tokenizer files**
- **Config files**
- **Model weights**
- **Model card** — intended use, limits, languages, settings
- **License info**
- **Training & evaluation** details

> ⚠️ A model card is **not a substitute** for your own evaluation.

---
## 🔍 3. When Should You Fine-Tune?

Fine-tuning is **one of several** customization options.

| Requirement | Good starting approach |
| --- | --- |
| Improve instructions | **Prompt engineering** |
| Answer from private / changing docs | **RAG** |
| Use current external info | **Retrieval / API integration** |
| Produce valid structured data | **Schema-constrained output + validation** |
| Execute business operations | **Tool calling + app logic** |
| Consistent behavior on specialized task | **Evaluate fine-tuning** |
| New foundation model | **Pretraining (much larger scale)** |

### RAG vs Fine-Tuning

- **RAG** → supplies relevant info at **inference time**. Great for **changing** documents & **citations**.
- **Fine-tuning** → **changes model behavior** through training. Great for **consistent task patterns / styles**.
- They can be **combined** (fine-tuned format + RAG for current facts).

> ⚠️ Fine-tuning is **not reliable** for keeping **frequently changing facts** up to date — use **retrieval** or a **trusted API**.

---
## 📚 4. Prepare a Training Dataset

**Dataset quality** is one of the most important factors in fine-tuning.

For **supervised instruction tuning**, a dataset contains **instruction + response** pairs.

### Example `data/train.jsonl`

```json
{"instruction":"Explain RAG in simple English.","response":"Retrieval-Augmented Generation finds relevant information and gives it to an LLM so the model can produce an answer grounded in that information."}
{"instruction":"Explain embeddings in one sentence.","response":"Embeddings are numerical vectors that represent information such as text in a form useful for measuring similarity."}
{"instruction":"Explain tool calling in simple English.","response":"Tool calling lets a model request that an application run a defined function, such as searching a database or calling an API."}
```

> ⚠️ These 3 examples are **illustrative only** — a real dataset needs **hundreds to thousands** of high-quality examples.

### Dataset checklist

- ✅ Represent the **real task**
- ✅ **Remove duplicates** & inconsistencies
- ✅ **No secrets / PII**
- ✅ **Standardized** formatting
- ✅ **Valid JSONL** records
- ✅ **No incorrect behaviors**
- ✅ **Train / eval split**
- ✅ **Version** the dataset
- ✅ **Document** provenance & permissions

### Data splits

| Split | Purpose |
| --- | --- |
| **Train** | Updates model / adapter weights |
| **Validation** | Chooses settings, detects overfitting |
| **Test** | Final held-out evaluation |

> ⚠️ Don't tune against test — keep it **separate** until the end.

In [ ]:
# Create sample train/val JSONL files for the demo
DATA_DIR = SCRIPT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_examples = [
    {'instruction': 'What is RAG?',
     'response': 'RAG retrieves relevant external information and supplies it to a language model to help ground its answer.'},
    {'instruction': 'What is tool calling?',
     'response': 'Tool calling lets a language model request that an application execute a defined function.'},
    {'instruction': 'What is an embedding?',
     'response': 'An embedding is a numerical vector representation of information such as text.'},
    {'instruction': 'What is a tokenizer?',
     'response': 'A tokenizer splits text into tokens that a language model can process.'},
    {'instruction': 'What is LoRA?',
     'response': 'LoRA adds small trainable low-rank matrices to selected model layers while keeping the base weights frozen.'},
]

validation_examples = [
    {'instruction': 'What is a chat template?',
     'response': 'A chat template formats messages with role markers so a chat model receives the expected input structure.'},
    {'instruction': 'What is QLoRA?',
     'response': 'QLoRA combines a quantized base model with trainable LoRA adapters to reduce memory requirements.'},
]

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

write_jsonl(DATA_DIR / 'train.jsonl', train_examples)
write_jsonl(DATA_DIR / 'validation.jsonl', validation_examples)

print('Training examples :', len(train_examples))
print('Validation examples:', len(validation_examples))
print('Saved to:', DATA_DIR)

---
## 🔤 5. Tokenization & Chat Templates

LLMs process **tokens**, not words.

A **tokenizer** converts text → token IDs. The model learns to predict the **next token** in a sequence.

### Instruction-tuning example

```text
### Instruction:
Explain embeddings in simple English.

### Response:
Embeddings are numerical vectors that represent information.
```

> ⚠️ The **exact** format depends on the model's training template. Chat models often use a **chat template** with role and message-boundary tokens.

### Why formatting matters

- The model must learn the format you'll use **at inference**
- If the model uses a chat template → follow it (don't invent an incompatible one)
- Training systems often compute loss on the **assistant response**; prompt tokens may be masked out
- Exact masking depends on the trainer & model

### Tokenization pipeline

```text
Raw text
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Model (next-token prediction)
   ↓
Generated tokens
   ↓
Decoded text
```

---
## ⚙️ 6. Full Fine-Tuning vs LoRA vs QLoRA

### 6.1 Full Fine-Tuning
Updates **all or most** trainable parameters.

- ✅ Broad behavior adaptation
- ❌ High memory & compute
- ❌ Larger artifacts
- ❌ Higher overfitting / catastrophic-forgetting risk

### 6.2 LoRA (Low-Rank Adaptation)
Adds **small trainable low-rank matrices** to selected layers — base weights **stay frozen**.

- ✅ Fewer trainable parameters
- ✅ Lower optimizer-state memory
- ✅ Easy to store & compare adapters
- ✅ Effective for task-specific adaptation

### 6.3 QLoRA (Quantized LoRA)
**Quantized base model** + trainable LoRA adapters.

- ✅ Even lower memory than LoRA w/ full-precision base
- ❌ Still depends on model size, seq length, batch size, quantization method

### Comparison

| Method | Base Model | Trainable | Resources |
| --- | --- | --- | --- |
| Full FT | Full precision | Most/all params | Highest |
| LoRA | Frozen | Adapter params | Lower |
| QLoRA | Quantized | Adapter params | Lowest |

> ⚠️ LoRA/QLoRA **don't automatically** make every model trainable on every GPU. Verify **hardware & memory** first.

---
## 🧰 7. Hugging Face Fine-Tuning Ecosystem

| Library | Purpose |
| --- | --- |
| **Transformers** | Load pretrained models & tokenizers |
| **Datasets** | Load, transform, manage datasets |
| **PEFT** | Parameter-efficient methods (LoRA, QLoRA) |
| **TRL** | Training utilities (SFT, DPO, etc.) |
| **PyTorch** | Tensor ops & model training |
| **Accelerate** | Multi-hardware training |

### Install

```bash
pip install torch transformers datasets peft trl accelerate
```

> ⚠️ Hardware-specific install requirements vary — follow official PyTorch guidance for your OS and accelerator.

---
## 💻 8. Practical Lab — Fine-Tune a Small Model With LoRA

The following demonstrates the **general structure** of supervised fine-tuning with a small causal LM and LoRA.

> ⚠️ **Hardware note:** Even a small model requires suitable memory. Start small and verify the model's **license** and hardware requirements.

### Lab outline

1. Prepare **dataset** (train + validation JSONL)
2. Load **model & tokenizer**
3. Format examples with the **chat template**
4. Configure **LoRA**
5. Configure the **Trainer**
6. **Train**
7. **Save the adapter**

In [ ]:
# ---- LoRA fine-tuning template (requires GPU + libraries installed) ----
# This cell is intentionally guarded. Uncomment / run on a machine with GPU.

print('LoRA fine-tuning template')
print('=' * 40)
print()
print('Hardware requirement: at least ~4-8 GB free GPU memory for a tiny model.')
print('Required libraries: torch, transformers, datasets, peft, trl, accelerate')
print()
print('The template below is COMPLETE and can be uncommented when ready:')
print()

template = '''
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'

# 1. Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Load dataset
dataset = load_dataset('json', data_files={
    'train':      'data/train.jsonl',
    'validation': 'data/validation.jsonl',
})

# 3. Format examples using chat template
def format_example(example):
    messages = [
        {'role': 'user',      'content': example['instruction']},
        {'role': 'assistant', 'content': example['response']},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {'text': text}

formatted = dataset.map(format_example)

# 4. LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
)

# 5. Trainer config
training_args = SFTConfig(
    output_dir='./outputs/smollm2-lora',
    dataset_text_field='text',
    max_length=512,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted['train'],
    eval_dataset=formatted['validation'],
    peft_config=lora_config,
    processing_class=tokenizer,
)

# 6. Train
trainer.train()

# 7. Save the adapter
trainer.model.save_pretrained('./outputs/smollm2-lora/adapter')
tokenizer.save_pretrained('./outputs/smollm2-lora/adapter')
'''

print(template)
print('=' * 40)
print('⚠️ Model IDs, target modules, and trainer arguments vary by model.')
print('   Always verify the model card, license, and library versions.')

---
## 📈 9. Evaluate the Fine-Tuned Model

The main question is **not whether training completed** — it's whether the resulting model performs **better on the intended task**.

Evaluate base & fine-tuned on the **same held-out test examples**.

### Evaluation dimensions

| Dimension | What to Check |
| --- | --- |
| Task correctness | Completes the intended task |
| Instruction following | Follows required constraints |
| Format compliance | Produces the required structure |
| General capability | Has general behavior **deteriorated**? |
| Safety | Continues to follow safety rules |
| Latency | Meets your requirements |
| Resource usage | Memory & compute |

### Evaluation workflow

1. Save a **baseline** before fine-tuning
2. Prepare a **separate test** dataset
3. Run **baseline** on test examples
4. Run **fine-tuned** on the **same** examples
5. Calculate task-specific metrics
6. Inspect failures **manually**
7. Compare **quality, latency, resources**
8. Keep fine-tuned **only if it meaningfully improves**

> ⚠️ **Training loss alone** does not guarantee better real-world answers.

In [ ]:
# A scaffold for baseline vs fine-tuned comparison
def evaluate_model(model_name, cases, predict_fn):
    """Compute simple metrics for a set of eval cases.
    predict_fn(case) -> string answer."""
    scores = []
    for case in cases:
        try:
            output = predict_fn(case)
            # Simple substring match — replace with task-specific metric
            hit = case['expected_keyword'].lower() in output.lower()
            scores.append({'case': case['id'], 'hit': hit})
        except Exception as e:
            scores.append({'case': case['id'], 'hit': False, 'error': str(e)})
    hits = sum(1 for s in scores if s.get('hit'))
    return {
        'model': model_name,
        'total': len(cases),
        'hits': hits,
        'accuracy': hits / len(cases) if cases else 0.0,
        'details': scores,
    }

# Mock eval cases
cases = [
    {'id': 'q1', 'input': 'What is RAG?',         'expected_keyword': 'retriev'},
    {'id': 'q2', 'input': 'What is LoRA?',        'expected_keyword': 'low-rank'},
    {'id': 'q3', 'input': 'What is an embedding?','expected_keyword': 'vector'},
]

# Fake predict functions for illustration
def baseline_predict(case):
    return {
        'q1': 'RAG stands for Retrieval-Augmented Generation.',
        'q2': 'LoRA is a technique.',
        'q3': 'An embedding is a vector of numbers.',
    }[case['id']]

def finetuned_predict(case):
    return {
        'q1': 'RAG retrieves relevant documents and grounds the answer.',
        'q2': 'LoRA adds low-rank matrices to selected layers.',
        'q3': 'An embedding is a numerical vector representation.',
    }[case['id']]

print(json.dumps(evaluate_model('baseline',  cases, baseline_predict),  indent=2))
print(json.dumps(evaluate_model('finetuned', cases, finetuned_predict), indent=2))

---
## ⚠️ 10. Common Fine-Tuning Mistakes

| Mistake | Better Approach |
| --- | --- |
| Fine-tuning before testing prompts | Establish a **strong baseline** first |
| Using a tiny dataset as proof of quality | Treat tiny datasets as **pipeline smoke tests** |
| Training & testing on same examples | Keep **held-out** evaluation |
| Inconsistent answer formats | **Standardize** dataset & template |
| Ignoring model licensing | Verify **use & redistribution** conditions |
| Incompatible LoRA targets | **Inspect architecture** & docs |
| Overtraining on repeated examples | Monitor **validation** & generalization |
| Assuming FT stores private docs | Use **retrieval** for changing info |
| Ignoring inference hardware | **Benchmark** the serving config |
| Assuming adapter = standalone model | Retain the **compatible base model** |

---
## 🔐 11. Data, Licensing, & Model Safety

Before training or deploying a model, check:

- Can the dataset **legally** be used for training?
- Is **personal / confidential** info removed?
- Does the model license permit your **intended commercial use**?
- Is **redistribution** of weights or adapters allowed?
- Do training examples contain **biased or unsafe** behavior?
- Does the fine-tuned model introduce **new failure modes**?
- Does the deployment env **protect model files & user data**?

> ⚠️ A fine-tuned model is **not automatically** private, unbiased, safe, or compliant. Fine-tuning can **reproduce** patterns from training data — **dataset governance & evaluation matter**.

---
## 🏋️ 12. Final Project — Domain-Specific Fine-Tuned Assistant

### 🎯 Goal
Fine-tune a small instruction model for a **clearly defined task**, then evaluate whether it **improves** over the original model.

### Possible tasks

- Support-ticket classification
- Short technical explanations
- Structured info extraction
- Categorization of issue reports
- Domain-specific instruction following

### Core requirements

- [ ] Select a suitable model & verify its **license**
- [ ] Define a **measurable** task
- [ ] Create & validate an **instruction dataset**
- [ ] Separate **train / validation / test**
- [ ] Establish a **baseline** model evaluation
- [ ] Run a small **supervised fine-tuning** experiment
- [ ] Use **LoRA or another justified** method
- [ ] Save the trained **adapter**
- [ ] Evaluate the fine-tuned model on **held-out data**
- [ ] Compare quality against the **baseline**
- [ ] Inspect **failures & regressions**
- [ ] Document **hardware, training settings, limitations**
- [ ] Explain whether fine-tuning was **worth the cost**

### Suggested structure

```text
13-Fine-Tuning-and-Open-Models/
├── README.md
├── requirements.txt
├── .env.example
├── data/
│   ├── train.jsonl
│   ├── validation.jsonl
│   └── test.jsonl
├── src/
│   ├── prepare_dataset.py
│   ├── train.py
│   ├── evaluate.py
│   └── inference.py
├── configs/
│   └── lora_config.yaml
├── outputs/
│   └── .gitkeep
├── tests/
│   ├── test_dataset.py
│   └── test_output_format.py
└── reports/
    └── baseline_vs_finetuned.md
```

> ⚠️ **Do NOT commit** large model weights, private datasets, or training artifacts without clear permission.

### Stretch goals

1. Compare **LoRA vs QLoRA**
2. Compare **two small open-weight** models
3. Measure **trainable parameters & peak GPU memory**
4. Experiment with **learning rate & training duration**
5. Compare **adapter inference vs merged-model inference**
6. Package with **FastAPI**
7. Deploy behind an **authenticated** API
8. Track experiments with **MLflow**

---
## 🎤 13. Interview Questions

**Q1. What is fine-tuning?**  
Continuing to train a pretrained model on task-specific data to adapt its behavior.

**Q2. Pretraining vs fine-tuning?**  
Pretraining teaches broad capabilities from a large corpus. Fine-tuning adapts an existing model to a particular task, format, or behavior.

**Q3. What is an open-weight model?**  
A model whose trained parameters are made available under specified terms. **Open weights ≠ unrestricted or fully open-source** license.

**Q4. When to prefer RAG over fine-tuning?**  
When answers depend on **changing documents**, external facts, or sources that must be **cited**.

**Q5. What is LoRA?**  
Trainable **low-rank adapter matrices** added to selected layers while keeping base weights frozen.

**Q6. What is QLoRA?**  
A **quantized base model** + trainable LoRA adapters → lower memory.

**Q7. What is supervised fine-tuning?**  
Training a model on examples that specify the desired behavior — typically **instruction + response** pairs.

**Q8. Why a validation set?**  
To compare training settings & detect overfitting **without** relying only on training loss.

**Q9. Why a held-out test set?**  
Independent estimate of performance on **unseen** examples.

**Q10. Does lower training loss mean a better model?**  
**No.** The model may overfit, learn unwanted patterns, or fail to generalize.

**Q11. What is a LoRA adapter?**  
A small set of learned parameters that **modifies** a compatible base model's behavior without storing a full copy.

**Q12. Does fine-tuning guarantee factual accuracy?**  
**No.** Use evaluation, retrieval, validation, and other appropriate controls.

---
## ✅ 14. Module Completion Checklist

- [ ] Understand hosted APIs & open-weight models
- [ ] Explain when fine-tuning is appropriate
- [ ] Compare RAG, prompting, fine-tuning
- [ ] Prepare a valid instruction dataset
- [ ] Understand tokenization & chat templates
- [ ] Explain full fine-tuning, LoRA, QLoRA
- [ ] Load a pretrained model with Transformers
- [ ] Run a small fine-tuning experiment
- [ ] Save & reload the adapter
- [ ] Evaluate against a baseline using held-out data
- [ ] Check licensing, data governance, hardware
- [ ] Complete the **Domain-Specific Fine-Tuned Assistant**
- [ ] Document **real results and limitations**

---
## 🏁 15. Key Takeaways

- **Fine-tuning** adapts an existing model — it is **not** training from scratch
- **Open-weight** models provide more deployment flexibility, **subject to license & technical requirements**
- **High-quality, consistent** training examples matter more than sheer dataset size
- **LoRA & QLoRA** can reduce training resource requirements
- Fine-tuning should be justified through **measurable improvement over a baseline**
- **RAG** is often a better fit for changing facts & document-grounded answers
- **Training loss alone** is not enough to establish model quality
- **Data governance, licensing, hardware, deployment** are part of responsible fine-tuning

### 🧭 Golden Rule

> **Fine-tune only when you can measure improvement over a baseline. Otherwise, use prompting, RAG, or better data — they're often cheaper and more effective.**

---

### 🔗 What's Next?

**Module 14 — Multimodal GenAI**

- AI systems with **text, images, audio, video**
- **Multimodal model** concepts
- **Vision-language** applications
- **Speech** processing
- Combining different modalities in real applications